# Class DataPreprocessor

In [ ]:
import os
import json
import pickle
import logging
from typing import Optional, Dict, Any, List, Tuple, Union

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder


# =========================
#  CUSTOM EXCEPTIONS
# =========================

# class FileFormatError(Exception):
#     """Raised when file format is not supported."""
#     pass


# # (Có thể dùng luôn built-in FileNotFoundError; ở đây chỉ gói lại cho rõ.)
# class DataFileNotFoundError(FileNotFoundError):
#     """Raised when file path does not exist."""
#     pass


# =========================
#  DATA PREPROCESSOR CLASS
# =========================

class DataPreprocessor:
    """
    DataPreprocessor cho project y khoa (Pima, v.v.)
    Thực hiện:
      - Load data (.csv, .xlsx, .json)
      - Hidden missing (0 -> NaN cho các cột y khoa)
      - Missing value (median toàn tập hoặc theo nhóm)
      - Outlier (IQR) + winsorize hoặc flag
      - Encode categoricals
      - Scaling
      - Feature engineering
      - Lưu kết quả + fit_transform pipeline
    """

    # chọn ra các cột mà nếu chứa giá trị không là vô lý
    MEDICAL_ZERO_AS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

    # chọn các cột thường có outlier y khoa
    OUTLIER_COLS = ["Glucose", "BMI", "BloodPressure", "Insulin"]

    def __init__(self, config: Optional[Dict[str, Any]] = None):
        # config là một optional parameter, chứa các câu lệnh điều khiển class tiền xử lý này
            # nếu config khong được truyền vào, sẽ sử dụng giá trị mặc định là None
        
        """
        config có thể chứa một số key gợi ý:
          - target_col: str
          - missing_strategy: 'global_median' | 'group_age' | 'group_outcome'
          - outlier_strategy: 'winsorize' | 'flag'
          - scaler: 'standard' | 'minmax' | None
          - encoder_strategy: 'auto'
          - pregnancy_high_threshold: int (mặc định 3)
          - scaler_path: str (đường dẫn lưu scaler = pickle)
        """

        # tạo biến self.config để lưu lại tham số config được truyền vào ban đầu
            # nếu config không được truyền vào, sẽ khởi tạo self.config là một dict rỗng
        self.config = config.copy() if config is not None else {}

        # self.config.setdefault(key, value)
            # nếu key chưa có trong self.config (config được truyền vào), 
            # thì sẽ thêm key với giá trị value mặc định này vào self.config
        self.config.setdefault("target_col", "Outcome") # target_col là kết quả dự đoán
        self.config.setdefault("missing_strategy", "global_median") # missing_strategy là cách để xử lý giá trị thiếu
        self.config.setdefault("outlier_strategy", "winsorize") # outlier_strategy là cách để xử lý ngoại lệ
        self.config.setdefault("scaler", "standard") # scaler là phương pháp chuẩn hóa
        self.config.setdefault("encoder_strategy", "auto") # encoder_strategy là cách mã hóa
        self.config.setdefault("pregnancy_high_threshold", 3) # pregnancy_high_threshold là ngưỡng đánh dấu người có số lần mang thai cao
        self.config.setdefault("scaler_path", "artifacts/scaler.pkl") # scaler_path là đường dẫn để lưu scaler 
                                                                            # scaler là công cụ chuẩn hóa số liệu (đưa các giá trị về cùng một thang đo)                                                         
        
        
        #  I. INITIALIZATION                                                    

        # biến lưu DataFrame nội bộ trong class (khi dùng fit_transform) 
            # chứa dataframe hoặc None nếu chưa load dữ liệu
        self.df_: Optional[pd.DataFrame] = None

        # fit là quá trình tìm công thức chuẩn hóa số liệu
        # , transform là quá trình áp dụng công thức đó lên số liệu

        # Danh sách (list) chứa tên các cột dạng số (numeric)
        self.numeric_cols: List[str] = []
        # Danh sách chứa tên các cột dạng phân loại (categorical)
        self.categorical_cols: List[str] = []
        # Danh sách chứa tên các cột dạng thời gian (datetime)
        self.datetime_cols: List[str] = []

        # Lưu encoder và scaler để tái sử dụng
        self.encoders: Dict[str, Any] = {} # cái này là lable encoder(0, 1, 2,...)
            # ví dụ {'male': 0, 'female': 1}
        
        # chọn scaler để chuẩn hóa số liệu: StandardScaler hoặc MinMaxScaler
        self.scaler: Optional[Union[StandardScaler, MinMaxScaler]] = None

        # ====== LOGGING ======
        self.logger = logging.getLogger(self.__class__.__name__)
            # self.__class__.__name__ trả về tên của class, ví dụ "DataPreprocessor".
            # Kết quả là được một logger tên "DataPreprocessor".
            # mỗi class có thể có logger riêng biệt để dễ phân biệt khi log.
        if not self.logger.handlers:
            # Kiểm tra xem logger này đã có handler chưa.
            # Handler = đối tượng chịu trách nhiệm xuất log ra đâu (màn hình, file, v.v.).
            # Nếu chưa có → tạo mới (tránh log bị nhân đôi khi gọi class nhiều lần).
            handler = logging.StreamHandler()
                # Tạo handler để cho phép in log ra màn hình console (như print()).
            formatter = logging.Formatter(
                fmt="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
                datefmt="%Y-%m-%d %H:%M:%S",
            )
                # tạo Định dạng (formatter) cho nội dung log được in, ví dụ:
                # 2025-11-10 16:15:23 | INFO | DataPreprocessor | Khởi tạo DataPreprocessor với config:.....
                # %(asctime)s → thời gian
                # %(levelname)s → cấp độ log (INFO, WARNING, ERROR, …)
                # %(name)s → tên logger (ở đây là tên class)
                # %(message)s → nội dung log
            handler.setFormatter(formatter)
                # Gắn formatter cho handler → để có được log đúng định dạng vừa tạo.
            self.logger.addHandler(handler)
                # Gắn handler này vào logger.
                # → Từ giờ mỗi khi gọi self.logger.info(...), log sẽ được gửi tới handler đó.
        self.logger.setLevel(logging.INFO)
            # Thiết lập mức độ log tối thiểu mà logger sẽ ghi lại.
            # Ở đây là INFO, nên chỉ log INFO trở lên.
        self.logger.info("Khởi tạo DataPreprocessor với config:")
            # In ra log thông tin (cấp độ INFO) với message là "Khởi tạo DataPreprocessor với config:"
        self.logger.info(self.config)
            # In ra log thông tin (cấp độ INFO) với message là nội dung của self.config.

    def __repr__(self) -> str: # hàm để in ra config của class
        pretty_cfg = json.dumps(self.config, indent=2, default=str, ensure_ascii=False)
        return f"DataPreprocessor(config={pretty_cfg})"
    # nó sẽ in ra như vầy:
    # DataPreprocessor(config={
                #   "target_col": "Outcome",
                #   "missing_strategy": "global_median",
                #   "outlier_strategy": "winsorize",
                #   "scaler": "standard",
                #   "encoder_strategy": "auto",
                #   "pregnancy_high_threshold": 3,
                #   "scaler_path": "artifacts/scaler.pkl"
                # })

    # =========================
    #  HELPER: INFER COLUMN TYPES
    # =========================

    def _infer_column_types(self, df: pd.DataFrame) -> None:
        """Tự động phân loại cột: numeric / categorical / datetime."""

        self.numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        # đọc dataframe và lấy tên các cột có kiểu số

        self.datetime_cols = df.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns.tolist()
        # đọc dataframe và lấy tên các cột có kiểu datetime

        cat_candidates = df.select_dtypes(include=["object", "category"]).columns.tolist()
        # đọc dataframe và lấy tên các cột có kiểu object hoặc category
        for col in df.columns:
            # Những cột numeric mà số lượng các giá trị <= 10, cũng có thể coi như categorical
            if col not in cat_candidates and col not in self.datetime_cols:
                if df[col].dtype != "O" and df[col].nunique(dropna=True) <= 10:
                    cat_candidates.append(col)

        self.categorical_cols = sorted(list(set(cat_candidates) - set(self.datetime_cols)))
            # loại bỏ tên cột mà bị trùng lặp hoặc là tên các cột datetime khỏi categorical

        self.logger.info(f"Phân loại cột:")
        self.logger.info(f"  Numeric     : {self.numeric_cols}")
        self.logger.info(f"  Categorical : {self.categorical_cols}")
        self.logger.info(f"  Datetime    : {self.datetime_cols}")
        # in ra thông tin phân loại cột, ví dụ:
            # 2025-11-10 16:45:22 | INFO | DataPreprocessor | Phân loại cột:
            # 2025-11-10 16:45:22 | INFO | DataPreprocessor | Numeric : ['Age', 'Glucose', 'BMI']
            # 2025-11-10 16:45:22 | INFO | DataPreprocessor | Categorical : ['Gender', 'Outcome']
            # 2025-11-10 16:45:22 | INFO | DataPreprocessor | Datetime : ['DateMeasured']

    # =========================
    #  II. LOAD DATA
    # =========================

    def load_data(self, path: str) -> pd.DataFrame:
        """
        Hỗ trợ: .csv, .xlsx, .json
        Dùng custom exceptions: FileFormatError, DataFileNotFoundError
        """
        self.logger.info(f"Đọc dữ liệu từ: {path}")

        if not os.path.exists(path):
            self.logger.error(f"File không tồn tại: {path}")
            raise DataFileNotFoundError(f"File not found: {path}")

        ext = os.path.splitext(path)[1].lower()

        try:
            if ext == ".csv":
                df = pd.read_csv(path)
            elif ext in [".xlsx", ".xls"]:
                df = pd.read_excel(path)
            elif ext == ".json":
                df = pd.read_json(path)
            else:
                self.logger.error(f"Định dạng file không được hỗ trợ: {ext}")
                raise FileFormatError(f"Unsupported file format: {ext}")
        except Exception as e:
            self.logger.exception(f"Lỗi khi đọc file: {e}")
            raise

        self.logger.info(f"Đọc thành công: shape = {df.shape}")
        self._infer_column_types(df)
        self.df_ = df
        return df

    # =========================
    #  III. HIDDEN MISSING
    # =========================

    def detect_hidden_missing(self, df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
        """
        - Phát hiện NaN thường
        - 0 bất hợp lý -> NaN cho các cột y khoa
        """
        if df is None:
            if self.df_ is None:
                raise ValueError("Chưa có DataFrame. Hãy truyền df hoặc gọi load_data().")
            df = self.df_

        df = df.copy()
        self.logger.info("Phát hiện hidden missing (0 -> NaN) cho các cột y khoa...")

        for col in self.MEDICAL_ZERO_AS_MISSING:
            if col in df.columns:
                zero_count = (df[col] == 0).sum()
                self.logger.info(f"  Cột {col}: số lượng 0 (bất hợp lý) = {zero_count}")
                df.loc[df[col] == 0, col] = np.nan

        self.logger.info("Hoàn thành detect_hidden_missing.")
        self.df_ = df
        return df

    # =========================
    #  IV. HANDLE MISSING
    # =========================

    def handle_missing(self, df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
        """
        - Chiến lược 1: median toàn tập
        - Chiến lược 2: median theo nhóm Age_group hoặc Outcome
        """
        if df is None:
            if self.df_ is None:
                raise ValueError("Chưa có DataFrame. Hãy truyền df hoặc gọi load_data().")
            df = self.df_

        df = df.copy()
        missing_before = df.isna().sum()
        self.logger.info("Missing values trước khi xử lý:")
        self.logger.info(missing_before[missing_before > 0])

        strategy = self.config.get("missing_strategy", "global_median")
        target_col = self.config.get("target_col", "Outcome")

        # Nếu cần Age_group mà chưa có thì tạo tạm đơn giản
        if strategy == "group_age" and "Age_group" not in df.columns and "Age" in df.columns:
            df["Age_group"] = pd.cut(
                df["Age"],
                bins=[20, 30, 40, 50, 120],
                labels=["21-30", "31-40", "41-50", "50+"],
                right=True,
                include_lowest=True,
            )
            self.logger.info("Tạo tạm Age_group để impute theo nhóm tuổi.")

        if strategy == "global_median":
            self.logger.info("Chiến lược missing: median toàn bộ tập.")
            for col in self.numeric_cols:
                if df[col].isna().any():
                    median_val = df[col].median()
                    df[col].fillna(median_val, inplace=True)

        elif strategy in ["group_age", "group_outcome"]:
            if strategy == "group_age":
                group_col = "Age_group"
            else:
                group_col = target_col

            if group_col not in df.columns:
                raise ValueError(f"Không tìm thấy cột group '{group_col}' để impute theo nhóm.")

            self.logger.info(f"Chiến lược missing: median theo nhóm {group_col}.")
            for col in self.numeric_cols:
                if df[col].isna().any():
                    medians = df.groupby(group_col)[col].transform("median")
                    df[col] = df[col].fillna(medians)

        else:
            raise ValueError(f"Unknown missing_strategy: {strategy}")

        missing_after = df.isna().sum()
        self.logger.info("Missing values sau khi xử lý:")
        self.logger.info(missing_after[missing_after > 0])

        self.df_ = df
        return df

    # =========================
    #  V. OUTLIERS (IQR)
    # =========================

    @staticmethod
    def detect_outliers_series(series: pd.Series) -> Tuple[pd.Series, float, float]:
        """
        Trả về:
          - mask: True nếu là outlier theo IQR rule.
          - lower_bound, upper_bound
        """
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        mask = (series < lower) | (series > upper)
        return mask, lower, upper

    def detect_outliers(self, df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
        """
        - Áp dụng IQR cho các cột OUTLIER_COLS.
        - outlier_strategy:
            * 'winsorize': đẩy về [lower, upper]
            * 'flag': thêm cột is_outlier_*
        """
        if df is None:
            if self.df_ is None:
                raise ValueError("Chưa có DataFrame. Hãy truyền df hoặc gọi load_data().")
            df = self.df_

        df = df.copy()
        strategy = self.config.get("outlier_strategy", "winsorize")

        self.logger.info(f"Phát hiện & xử lý outlier với chiến lược: {strategy}")

        for col in self.OUTLIER_COLS:
            if col not in df.columns:
                continue

            mask, lower, upper = self.detect_outliers_series(df[col].dropna())
            # align mask với df (chỗ NaN thì False)
            full_mask = df[col].copy()
            full_mask[:] = False
            full_mask.loc[mask.index] = mask
            outlier_count = full_mask.sum()

            self.logger.info(
                f"  Cột {col}: outliers = {int(outlier_count)}, "
                f"lower={lower:.3f}, upper={upper:.3f}"
            )

            if strategy == "winsorize":
                df[col] = df[col].clip(lower=lower, upper=upper)
            elif strategy == "flag":
                flag_col = f"is_outlier_{col}"
                df[flag_col] = full_mask.astype(int)
            else:
                raise ValueError(f"Unknown outlier_strategy: {strategy}")

        self.df_ = df
        return df

    # =========================
    #  VI. ENCODE CATEGORICALS
    # =========================

    @staticmethod
    def encode_column(series: pd.Series) -> Tuple[np.ndarray, LabelEncoder]:
        """
        Demo staticmethod: label encode cho 1 series.
        Trả về (encoded_values, encoder).
        """
        le = LabelEncoder()
        encoded = le.fit_transform(series.astype(str))
        return encoded, le

    def encode_categoricals(self, df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
        """
        - Tự động phát hiện cột object/category.
        - Binary -> LabelEncoder
        - Nhiều mức -> get_dummies (OneHot encoder đơn giản bằng pandas)
        """
        if df is None:
            if self.df_ is None:
                raise ValueError("Chưa có DataFrame. Hãy truyền df hoặc gọi load_data().")
            df = self.df_

        df = df.copy()
        self.logger.info("Mã hóa biến phân loại...")

        encoded_cols = []

        for col in self.categorical_cols:
            if col not in df.columns:
                continue

            unique_vals = df[col].dropna().unique()
            n_unique = len(unique_vals)

            if n_unique == 0:
                continue

            if n_unique <= 2:
                # Binary -> LabelEncoder (dùng staticmethod)
                self.logger.info(f"  Binary column (LabelEncoder): {col}")
                encoded, le = self.encode_column(df[col])
                new_col = f"{col}_LE"
                df[new_col] = encoded
                self.encoders[col] = le
                encoded_cols.append(new_col)
                # Có thể giữ cột gốc để tham khảo, hoặc drop:
                # df.drop(columns=[col], inplace=True)
            else:
                # Multi-level -> OneHot bằng pandas.get_dummies
                self.logger.info(f"  Multi-level column (OneHot): {col}")
                dummies = pd.get_dummies(df[col], prefix=col, dummy_na=False)
                df = pd.concat([df, dummies], axis=1)
                encoded_cols.extend(dummies.columns.tolist())
                # df.drop(columns=[col], inplace=True)

        self.logger.info(f"Các cột được mã hóa / one-hot: {encoded_cols}")
        self.df_ = df
        return df

    # =========================
    #  VII. SCALING
    # =========================

    def scale_features(self, df: Optional[pd.DataFrame] = None, fit: bool = True) -> pd.DataFrame:
        """
        - scaler: 'standard' hoặc 'minmax' hoặc None.
        - Lưu scaler bằng pickle để dùng khi deploy.
        """
        if df is None:
            if self.df_ is None:
                raise ValueError("Chưa có DataFrame. Hãy truyền df hoặc gọi load_data().")
            df = self.df_

        df = df.copy()
        scaler_name = self.config.get("scaler", "standard")

        if scaler_name is None:
            self.logger.info("Không dùng scaler (scaler=None).")
            self.df_ = df
            return df

        if scaler_name == "standard":
            scaler_cls = StandardScaler
        elif scaler_name == "minmax":
            scaler_cls = MinMaxScaler
        else:
            raise ValueError(f"Unknown scaler: {scaler_name}")

        numeric_cols = [col for col in self.numeric_cols if col in df.columns]

        if fit or self.scaler is None:
            self.logger.info(f"Fit scaler ({scaler_name}) cho các cột numeric: {numeric_cols}")
            self.scaler = scaler_cls()
            df[numeric_cols] = self.scaler.fit_transform(df[numeric_cols])
            # Lưu scaler
            scaler_path = self.config.get("scaler_path", "artifacts/scaler.pkl")
            os.makedirs(os.path.dirname(scaler_path), exist_ok=True)
            with open(scaler_path, "wb") as f:
                pickle.dump(self.scaler, f)
            self.logger.info(f"Lưu scaler vào: {scaler_path}")
        else:
            self.logger.info(f"Transform với scaler đã fit cho các cột numeric: {numeric_cols}")
            df[numeric_cols] = self.scaler.transform(df[numeric_cols])

        # Log mean/std sau scaling
        means = df[numeric_cols].mean().round(3)
        stds = df[numeric_cols].std().round(3)
        self.logger.info("Trung bình sau scaling (xấp xỉ):")
        self.logger.info(means)
        self.logger.info("Độ lệch chuẩn sau scaling (xấp xỉ):")
        self.logger.info(stds)

        self.df_ = df
        return df

    # =========================
    #  VIII. FEATURE ENGINEERING
    # =========================

    def feature_engineering(self, df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
        """
        - BMI_category
        - Age_group
        - Pregnancy_high
        - Glucose * BMI, Insulin / Glucose
        """
        if df is None:
            if self.df_ is None:
                raise ValueError("Chưa có DataFrame. Hãy truyền df hoặc gọi load_data().")
            df = self.df_

        df = df.copy()
        new_features = []

        # BMI_category
        if "BMI" in df.columns:
            bins = [0, 18.5, 25, 30, np.inf]
            labels = ["Underweight", "Normal", "Overweight", "Obese"]
            df["BMI_category"] = pd.cut(df["BMI"], bins=bins, labels=labels, right=False)
            new_features.append("BMI_category")
            self.categorical_cols.append("BMI_category")

        # Age_group
        if "Age" in df.columns:
            df["Age_group"] = pd.cut(
                df["Age"],
                bins=[20, 30, 40, 50, 120],
                labels=["21-30", "31-40", "41-50", "50+"],
                right=True,
                include_lowest=True,
            )
            if "Age_group" not in self.categorical_cols:
                self.categorical_cols.append("Age_group")
            new_features.append("Age_group")

        # Pregnancy_high
        if "Pregnancies" in df.columns:
            thr = self.config.get("pregnancy_high_threshold", 3)
            df["Pregnancy_high"] = (df["Pregnancies"] >= thr).astype(int)
            new_features.append("Pregnancy_high")

        # Interaction: Glucose * BMI
        if "Glucose" in df.columns and "BMI" in df.columns:
            df["Glucose_BMI"] = df["Glucose"] * df["BMI"]
            new_features.append("Glucose_BMI")

        # Interaction: Insulin / Glucose
        if "Insulin" in df.columns and "Glucose" in df.columns:
            df["Insulin_over_Glucose"] = df["Insulin"] / df["Glucose"].replace(0, np.nan)
            new_features.append("Insulin_over_Glucose")

        self.logger.info(f"Created features: {new_features}")

        # Optional: quick correlation with target nếu có
        target_col = self.config.get("target_col", "Outcome")

        if target_col in df.columns:
        # Chỉ lấy những feature mới có kiểu numeric
            numeric_new_feats = [
                c for c in new_features 
                if pd.api.types.is_numeric_dtype(df[c])
            ]

        if numeric_new_feats:
            cols_for_corr = numeric_new_feats + [target_col]
            corr = df[cols_for_corr].corr(numeric_only=True)[target_col].sort_values(ascending=False)
            self.logger.info("Tương quan thô giữa feature mới (numeric) và target:")
            self.logger.info(corr)
        else:
            self.logger.info("Không có feature mới nào dạng numeric để tính tương quan.")


        self.df_ = df
        return df

    # =========================
    #  IX. SAVE PROCESSED
    # =========================

    def save_processed(self, df: Optional[pd.DataFrame], path: str) -> None:
        if df is None:
            if self.df_ is None:
                raise ValueError("Chưa có DataFrame. Hãy truyền df hoặc gọi load_data().")
            df = self.df_

        os.makedirs(os.path.dirname(path), exist_ok=True)
        df.to_csv(path, index=False)
        self.logger.info(f"Lưu kết quả xử lý ra {path} | shape = {df.shape}")

    # =========================
    #  X. FIT_TRANSFORM PIPELINE
    # =========================

    def fit_transform(self, data_or_path: Union[str, pd.DataFrame]) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Pipeline đầy đủ:
          1) load_data (nếu input là path)
          2) detect_hidden_missing
          3) handle_missing
          4) detect_outliers
          5) feature_engineering
          6) encode_categoricals
          7) scale_features
        Trả về:
          - df_processed
          - df_summary (describe)
        """
        self.logger.info("=== BẮT ĐẦU PREPROCESSING PIPELINE ===")

        # 1) Load hoặc dùng df
        if isinstance(data_or_path, str):
            df = self.load_data(data_or_path)
        else:
            df = data_or_path.copy()
            self._infer_column_types(df)
            self.df_ = df

        # 2) Hidden missing
        df = self.detect_hidden_missing(df)

        # 3) Missing
        df = self.handle_missing(df)

        # 4) Outliers
        df = self.detect_outliers(df)

        # 5) Feature Engineering
        df = self.feature_engineering(df)

        # Update lại type sau khi tạo feature mới
        self._infer_column_types(df)

        # 6) Encode categoricals
        df = self.encode_categoricals(df)

        # 7) Scaling
        df = self.scale_features(df, fit=True)

        # Summary
        summary = df.describe(include="all")
        self.logger.info("=== TÓM TẮT DATA SAU PREPROCESSING (describe) ===")
        self.logger.info(summary)

        self.logger.info("=== KẾT THÚC PREPROCESSING PIPELINE ===")
        self.df_ = df
        return df, summary


In [14]:
# ======================================================================
# Cell 2 - CẤU HÌNH & KHỞI TẠO DataPreprocessor (I)
# ======================================================================

config = {
    "target_col": "Outcome",
    "missing_strategy": "group_age",    # hoặc 'global_median', 'group_outcome'
    "outlier_strategy": "winsorize",    # hoặc 'flag'
    "scaler": "standard",               # hoặc 'minmax' hoặc None
    "encoder_strategy": "auto",
    "pregnancy_high_threshold": 3,
    "scaler_path": "artifacts/scaler.pkl"
}

pre = DataPreprocessor(config)

# __repr__ sẽ in config gọn gàng
print(pre)


2025-11-10 11:48:13 | INFO | DataPreprocessor | Khởi tạo DataPreprocessor với config:
2025-11-10 11:48:13 | INFO | DataPreprocessor | {'target_col': 'Outcome', 'missing_strategy': 'group_age', 'outlier_strategy': 'winsorize', 'scaler': 'standard', 'encoder_strategy': 'auto', 'pregnancy_high_threshold': 3, 'scaler_path': 'artifacts/scaler.pkl'}


DataPreprocessor(config={
  "target_col": "Outcome",
  "missing_strategy": "group_age",
  "outlier_strategy": "winsorize",
  "scaler": "standard",
  "encoder_strategy": "auto",
  "pregnancy_high_threshold": 3,
  "scaler_path": "artifacts/scaler.pkl"
})


In [15]:
# ======================================================================
# Cell 3 - ĐỌC DỮ LIỆU + TỰ ĐỘNG PHÂN LOẠI CỘT (II)
# ======================================================================

DATA_PATH = r"raw_data/diabetes.csv"  # nếu chạy từ root project

df_raw = pre.load_data(DATA_PATH)

print(">>> 5 dòng đầu của dữ liệu gốc:")
display(df_raw.head())

print("\n>>> Kiểu dữ liệu từng cột:")
print(df_raw.dtypes)

print("\n>>> Các cột numeric / categorical / datetime sau khi _infer_column_types:")
print("Numeric     :", pre.numeric_cols)
print("Categorical :", pre.categorical_cols)
print("Datetime    :", pre.datetime_cols)

# ---------------------------------------------------------
# (TUỲ CHỌN) DEMO BẮT LỖI FILE KHÔNG TỒN TẠI
# ---------------------------------------------------------
try:
    pre.load_data("raw_data/khong_ton_tai.csv")
except DataFileNotFoundError as e:
    print("\n>>> Bắt được DataFileNotFoundError:", e)

# ---------------------------------------------------------
# (TUỲ CHỌN) DEMO BẮT LỖI ĐỊNH DẠNG FILE KHÔNG HỖ TRỢ
#   - Cần có 1 file .txt thật để test (ví dụ tạo file trống).
# ---------------------------------------------------------
try:
    pre.load_data("raw_data/diabetes.txt")  # nếu có file .txt
except FileFormatError as e:
    print("\n>>> Bắt được FileFormatError:", e)
except DataFileNotFoundError as e:
    print("\n>>> File .txt không tồn tại, nên ném DataFileNotFoundError trước:", e)


2025-11-10 11:48:13 | INFO | DataPreprocessor | Đọc dữ liệu từ: raw_data/diabetes.csv
2025-11-10 11:48:13 | INFO | DataPreprocessor | Đọc thành công: shape = (768, 9)
2025-11-10 11:48:13 | INFO | DataPreprocessor | Phân loại cột:
2025-11-10 11:48:13 | INFO | DataPreprocessor |   Numeric     : ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
2025-11-10 11:48:13 | INFO | DataPreprocessor |   Categorical : ['Outcome']
2025-11-10 11:48:13 | INFO | DataPreprocessor |   Datetime    : []


>>> 5 dòng đầu của dữ liệu gốc:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


2025-11-10 11:48:13 | INFO | DataPreprocessor | Đọc dữ liệu từ: raw_data/khong_ton_tai.csv
2025-11-10 11:48:13 | ERROR | DataPreprocessor | File không tồn tại: raw_data/khong_ton_tai.csv
2025-11-10 11:48:14 | INFO | DataPreprocessor | Đọc dữ liệu từ: raw_data/diabetes.txt
2025-11-10 11:48:14 | ERROR | DataPreprocessor | File không tồn tại: raw_data/diabetes.txt



>>> Kiểu dữ liệu từng cột:
Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object

>>> Các cột numeric / categorical / datetime sau khi _infer_column_types:
Numeric     : ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
Categorical : ['Outcome']
Datetime    : []

>>> Bắt được DataFileNotFoundError: File not found: raw_data/khong_ton_tai.csv

>>> File .txt không tồn tại, nên ném DataFileNotFoundError trước: File not found: raw_data/diabetes.txt


In [16]:
# ======================================================================
# Cell 4 - PHÁT HIỆN NaN & HIDDEN MISSING (III)
# ======================================================================

print(">>> Số lượng 0 'bất hợp lý' trước khi chuyển thành NaN:")
for col in pre.MEDICAL_ZERO_AS_MISSING:
    if col in df_raw.columns:
        zero_count = (df_raw[col] == 0).sum()
        print(f"  {col}: {zero_count}")

df_hidden = pre.detect_hidden_missing(df_raw)

print("\n>>> Số lượng NaN sau khi convert 0 -> NaN ở các cột y khoa:")
for col in pre.MEDICAL_ZERO_AS_MISSING:
    if col in df_hidden.columns:
        nan_count = df_hidden[col].isna().sum()
        print(f"  {col}: {nan_count}")


2025-11-10 11:48:14 | INFO | DataPreprocessor | Phát hiện hidden missing (0 -> NaN) cho các cột y khoa...
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Cột Glucose: số lượng 0 (bất hợp lý) = 5
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Cột BloodPressure: số lượng 0 (bất hợp lý) = 35
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Cột SkinThickness: số lượng 0 (bất hợp lý) = 227
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Cột Insulin: số lượng 0 (bất hợp lý) = 374
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Cột BMI: số lượng 0 (bất hợp lý) = 11
2025-11-10 11:48:14 | INFO | DataPreprocessor | Hoàn thành detect_hidden_missing.


>>> Số lượng 0 'bất hợp lý' trước khi chuyển thành NaN:
  Glucose: 5
  BloodPressure: 35
  SkinThickness: 227
  Insulin: 374
  BMI: 11

>>> Số lượng NaN sau khi convert 0 -> NaN ở các cột y khoa:
  Glucose: 5
  BloodPressure: 35
  SkinThickness: 227
  Insulin: 374
  BMI: 11


In [17]:
# ======================================================================
# Cell 5 - XỬ LÝ MISSING VALUES (IV) - median theo nhóm Age_group
# ======================================================================

print(">>> Số lượng NaN trước khi xử lý missing:")
print(df_hidden.isna().sum())

df_no_missing = pre.handle_missing(df_hidden)

print("\n>>> Số lượng NaN sau khi xử lý missing:")
print(df_no_missing.isna().sum())


2025-11-10 11:48:14 | INFO | DataPreprocessor | Missing values trước khi xử lý:
2025-11-10 11:48:14 | INFO | DataPreprocessor | Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64
2025-11-10 11:48:14 | INFO | DataPreprocessor | Tạo tạm Age_group để impute theo nhóm tuổi.
2025-11-10 11:48:14 | INFO | DataPreprocessor | Chiến lược missing: median theo nhóm Age_group.
C:\Users\HP\AppData\Local\Temp\ipykernel_19824\739487835.py:241: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  medians = df.groupby(group_col)[col].transform("median")
2025-11-10 11:48:14 | INFO | DataPreprocessor | Missing values sau khi xử lý:
2025-11-10 11:48:14 | INFO | DataPreprocessor | Series([], dtype: int64)


>>> Số lượng NaN trước khi xử lý missing:
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64

>>> Số lượng NaN sau khi xử lý missing:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
Age_group                   0
dtype: int64


In [18]:
# ======================================================================
# Cell 6 - PHÁT HIỆN & XỬ LÝ OUTLIER (V)
# ======================================================================

# Demo @staticmethod detect_outliers_series trên 1 cột (ví dụ Glucose)
if "Glucose" in df_no_missing.columns:
    mask_glu, lower_glu, upper_glu = DataPreprocessor.detect_outliers_series(df_no_missing["Glucose"])
    print(">>> Glucose - IQR bounds:")
    print("  lower:", lower_glu, " | upper:", upper_glu)
    print("  Số outlier (Glucose) theo IQR:", mask_glu.sum())

# Áp dụng cho toàn bộ các cột OUTLIER_COLS với chiến lược 'winsorize'
df_no_outlier = pre.detect_outliers(df_no_missing)

print("\n>>> Xem nhanh thống kê sau khi winsorize outliers:")
for col in pre.OUTLIER_COLS:
    if col in df_no_outlier.columns:
        print(f"\n--- {col} ---")
        print(df_no_outlier[col].describe())


2025-11-10 11:48:14 | INFO | DataPreprocessor | Phát hiện & xử lý outlier với chiến lược: winsorize
C:\Users\HP\AppData\Local\Temp\ipykernel_19824\739487835.py:297: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  full_mask[:] = False
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Cột Glucose: outliers = 0, lower=39.000, upper=201.000
C:\Users\HP\AppData\Local\Temp\ipykernel_19824\739487835.py:297: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  full_mask[:] = False
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Cột BMI: outliers = 8, lower=13.850, upper=50.250
C:\Users\HP\AppData\Local\Temp\ipykernel_19824\739487835.py:297: Futu

>>> Glucose - IQR bounds:
  lower: 39.0  | upper: 201.0
  Số outlier (Glucose) theo IQR: 0

>>> Xem nhanh thống kê sau khi winsorize outliers:

--- Glucose ---
count    768.000000
mean     121.651042
std       30.443537
min       44.000000
25%       99.750000
50%      117.000000
75%      140.250000
max      199.000000
Name: Glucose, dtype: float64

--- BMI ---
count    768.000000
mean      32.380469
std        6.668704
min       18.200000
25%       27.500000
50%       32.000000
75%       36.600000
max       50.250000
Name: BMI, dtype: float64

--- BloodPressure ---
count    768.000000
mean      72.303385
std       11.730031
min       40.000000
25%       64.000000
50%       72.000000
75%       80.000000
max      104.000000
Name: BloodPressure, dtype: float64

--- Insulin ---
count    768.000000
mean     133.035156
std       53.271457
min       22.500000
25%      105.000000
50%      131.000000
75%      160.000000
max      242.500000
Name: Insulin, dtype: float64


In [19]:
# ======================================================================
# Cell 7 - FEATURE ENGINEERING (VIII)
# ======================================================================

df_fe = pre.feature_engineering(df_no_outlier)

print(">>> Các cột mới sau feature_engineering:")
cols_new = [c for c in df_fe.columns if c in ["BMI_category", "Age_group", "Pregnancy_high", 
                                             "Glucose_BMI", "Insulin_over_Glucose"]]
print(cols_new)

print("\n>>> 5 dòng đầu với các feature mới:")
display(df_fe[cols_new + ["Outcome"]].head())

# (Nếu có target 'Outcome', logger đã in correlation thô rồi.)


2025-11-10 11:48:14 | INFO | DataPreprocessor | Created features: ['BMI_category', 'Age_group', 'Pregnancy_high', 'Glucose_BMI', 'Insulin_over_Glucose']
2025-11-10 11:48:14 | INFO | DataPreprocessor | Tương quan thô giữa feature mới (numeric) và target:
2025-11-10 11:48:14 | INFO | DataPreprocessor | Outcome                 1.000000
Glucose_BMI             0.522968
Pregnancy_high          0.196340
Insulin_over_Glucose   -0.020778
Name: Outcome, dtype: float64


>>> Các cột mới sau feature_engineering:
['Age_group', 'BMI_category', 'Pregnancy_high', 'Glucose_BMI', 'Insulin_over_Glucose']

>>> 5 dòng đầu với các feature mới:


,Age_group,BMI_category,Pregnancy_high,Glucose_BMI,Insulin_over_Glucose,Outcome
0,41-50,Obese,1,4972.8,0.885135,1
1,31-40,Overweight,0,2261.0,1.647059,0
2,31-40,Normal,1,4263.9,0.765027,1
3,21-30,Overweight,0,2500.9,1.056180,0
4,31-40,Obese,0,5904.7,1.226277,1


In [20]:
# ======================================================================
# Cell 8 - MÃ HÓA BIẾN PHÂN LOẠI (VI)
# ======================================================================

# Nhớ cập nhật lại phân loại cột (vì đã tạo thêm BMI_category, Age_group, ...)
pre._infer_column_types(df_fe)

print(">>> Categorical columns trước khi encode:")
print(pre.categorical_cols)

df_encoded = pre.encode_categoricals(df_fe)

print("\n>>> 5 dòng đầu sau khi encode (xem vài cột one-hot / label-encode):")
cols_preview = [c for c in df_encoded.columns 
                if "BMI_category" in c or "Age_group" in c or c.endswith("_LE")]
display(df_encoded[cols_preview + ["Outcome"]].head())


2025-11-10 11:48:14 | INFO | DataPreprocessor | Phân loại cột:
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Numeric     : ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome', 'Pregnancy_high', 'Glucose_BMI', 'Insulin_over_Glucose']
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Categorical : ['Age_group', 'BMI_category', 'Outcome', 'Pregnancy_high']
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Datetime    : []
2025-11-10 11:48:14 | INFO | DataPreprocessor | Mã hóa biến phân loại...
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Multi-level column (OneHot): Age_group
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Multi-level column (OneHot): BMI_category
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Binary column (LabelEncoder): Outcome
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Binary column (LabelEncoder): Pregnancy_high
2025-11-10 11:48:14 | INFO | DataPreprocessor | Các cột được mã 

>>> Categorical columns trước khi encode:
['Age_group', 'BMI_category', 'Outcome', 'Pregnancy_high']

>>> 5 dòng đầu sau khi encode (xem vài cột one-hot / label-encode):


,Age_group,BMI_category,Age_group_21-30,Age_group_31-40,Age_group_41-50,Age_group_50+,BMI_category_Underweight,BMI_category_Normal,BMI_category_Overweight,BMI_category_Obese,Outcome_LE,Pregnancy_high_LE,Outcome
0,41-50,Obese,False,False,True,False,False,False,False,True,1,1,1
1,31-40,Overweight,False,True,False,False,False,False,True,False,0,0,0
2,31-40,Normal,False,True,False,False,False,True,False,False,1,1,1
3,21-30,Overweight,True,False,False,False,False,False,True,False,0,0,0
4,31-40,Obese,False,True,False,False,False,False,False,True,1,0,1


In [21]:
# ======================================================================
# Cell 9 - CHUẨN HÓA DỮ LIỆU (VII)
# ======================================================================

df_scaled = pre.scale_features(df_encoded, fit=True)

print(">>> 5 dòng đầu sau khi scale (xem vài cột numeric):")
numeric_preview = pre.numeric_cols[:10]  # lấy vài cột đầu cho gọn
display(df_scaled[numeric_preview].head())

print("\n>>> Lưu ý:")
print("- Scaler đã được lưu vào file:", pre.config.get("scaler_path"))


2025-11-10 11:48:14 | INFO | DataPreprocessor | Fit scaler (standard) cho các cột numeric: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome', 'Pregnancy_high', 'Glucose_BMI', 'Insulin_over_Glucose']
2025-11-10 11:48:14 | INFO | DataPreprocessor | Lưu scaler vào: artifacts/scaler.pkl
2025-11-10 11:48:14 | INFO | DataPreprocessor | Trung bình sau scaling (xấp xỉ):
2025-11-10 11:48:14 | INFO | DataPreprocessor | Pregnancies                -0.0
Glucose                    -0.0
BloodPressure              -0.0
SkinThickness               0.0
Insulin                    -0.0
BMI                         0.0
DiabetesPedigreeFunction    0.0
Age                         0.0
Outcome                     0.0
Pregnancy_high              0.0
Glucose_BMI                 0.0
Insulin_over_Glucose        0.0
dtype: float64
2025-11-10 11:48:14 | INFO | DataPreprocessor | Độ lệch chuẩn sau scaling (xấp xỉ):
2025-11-10 11:48:14 | INFO | D

>>> 5 dòng đầu sau khi scale (xem vài cột numeric):


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Pregnancy_high
0,0.639947,0.866067,-0.025881,0.655077,-0.038228,0.182993,0.468492,1.425995,1.365896,0.912653
1,-0.844885,-1.204687,-0.537722,-0.022194,0.130828,-0.867370,-0.365061,-0.190672,-0.732120,-1.095707
2,1.233880,2.016485,-0.708335,0.316442,0.130828,-1.362542,0.604397,-0.105584,1.365896,0.912653
3,-0.844885,-1.073210,-0.537722,-0.699464,-0.733237,-0.642292,-0.920763,-1.041549,-0.732120,-1.095707
4,-1.141852,0.504506,-2.755699,0.655077,0.656780,1.608486,5.484909,-0.020496,1.365896,-1.095707



>>> Lưu ý:
- Scaler đã được lưu vào file: artifacts/scaler.pkl


In [22]:
# ======================================================================
# Cell 10 - CHẠY TOÀN BỘ PIPELINE + LƯU KẾT QUẢ (IX, X)
# ======================================================================

# Có thể dùng lại 'pre' hoặc tạo pre khác. Ở đây dùng lại:
df_processed, summary = pre.fit_transform(DATA_PATH)

print(">>> Hình dạng dữ liệu sau toàn bộ pipeline:")
print(df_processed.shape)

print("\n>>> Tóm tắt df.describe(include='all'):")
display(summary)

# Lưu kết quả đã xử lý
OUTPUT_PATH = r"processed_data/diabetes_processed.csv"
pre.save_processed(df_processed, OUTPUT_PATH)

print("\n>>> Đã lưu data sau khi preprocess vào:", OUTPUT_PATH)


2025-11-10 11:48:14 | INFO | DataPreprocessor | === BẮT ĐẦU PREPROCESSING PIPELINE ===
2025-11-10 11:48:14 | INFO | DataPreprocessor | Đọc dữ liệu từ: raw_data/diabetes.csv
2025-11-10 11:48:14 | INFO | DataPreprocessor | Đọc thành công: shape = (768, 9)
2025-11-10 11:48:14 | INFO | DataPreprocessor | Phân loại cột:
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Numeric     : ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Categorical : ['Outcome']
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Datetime    : []
2025-11-10 11:48:14 | INFO | DataPreprocessor | Phát hiện hidden missing (0 -> NaN) cho các cột y khoa...
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Cột Glucose: số lượng 0 (bất hợp lý) = 5
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Cột BloodPressure: số lượng 0 (bất hợp lý) = 35
2025-11-10 11:48:14 | INFO | DataPreprocessor |   Cột Sk

>>> Hình dạng dữ liệu sau toàn bộ pipeline:
(768, 24)

>>> Tóm tắt df.describe(include='all'):


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Age_group,...,Age_group_21-30,Age_group_31-40,Age_group_41-50,Age_group_50+,BMI_category_Underweight,BMI_category_Normal,BMI_category_Overweight,BMI_category_Obese,Outcome_LE,Pregnancy_high_LE
count,7.680000e+02,7.680000e+02,7.680000e+02,7.680000e+02,7.680000e+02,7.680000e+02,7.680000e+02,7.680000e+02,7.680000e+02,768,...,768,768,768,768,768,768,768,768,768.000000,768.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,...,2,2,2,2,2,2,2,2,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21-30,...,True,False,False,False,False,False,False,True,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,417,...,417,611,655,687,764,666,589,483,NaN,NaN
mean,-6.476301e-17,-1.757853e-16,-3.920475e-16,1.295260e-16,-6.938894e-18,2.035409e-16,2.451743e-16,1.931325e-16,7.401487e-17,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.348958,0.545573
std,1.000652e+00,1.000652e+00,1.000652e+00,1.000652e+00,1.000652e+00,1.000652e+00,1.000652e+00,1.000652e+00,1.000652e+00,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.476951,0.498243
min,-1.141852e+00,-2.552320e+00,-2.755699e+00,-2.505519e+00,-2.076294e+00,-2.127806e+00,-1.189553e+00,-1.041549e+00,-7.321202e-01,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000
25%,-8.448851e-01,-7.198675e-01,-7.083354e-01,-4.737072e-01,-5.266127e-01,-7.323236e-01,-6.889685e-01,-7.862862e-01,-7.321202e-01,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000
50%,-2.509521e-01,-1.528756e-01,-2.588085e-02,-2.219354e-02,-3.822840e-02,-5.709006e-02,-3.001282e-01,-3.608474e-01,-7.321202e-01,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,1.000000
75%,6.399473e-01,6.113310e-01,6.565737e-01,3.164417e-01,5.065079e-01,6.331487e-01,4.662269e-01,6.602056e-01,1.365896e+00,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000


2025-11-10 11:48:14 | INFO | DataPreprocessor | Lưu kết quả xử lý ra processed_data/diabetes_processed.csv | shape = (768, 24)



>>> Đã lưu data sau khi preprocess vào: processed_data/diabetes_processed.csv
